# Projet : analyse exploratoire d'une supply chain

Ce notebook est un **projet guidé** qui mobilise tout le module pandas sur un vrai jeu de données. Tu joues le rôle d'un ou d'une analyste qui vient d'arriver chez **DataCo Global**, une entreprise qui vend des articles de sport, de vêtements et d'électronique dans le monde entier. Ton responsable te transmet l'export des commandes de 2015 à début 2018 et te pose des questions très concrètes :

1. **Livrons-nous à l'heure ?** Et si non, où est le problème ?
2. **Quels produits comptent vraiment ?** (l'analyse ABC)
3. **Où gagne-t-on et où perd-on de l'argent ?**
4. **Y a-t-il de la fraude, et peut-on la repérer ?**
5. **Comment l'activité évolue-t-elle dans le temps ?**

Comme dans les autres notebooks : complète les cellules `# TODO`, compare avec le résultat attendu, et ne regarde la solution qu'après avoir vraiment essayé. Mais ici, il y a une étape de plus : après chaque analyse, une cellule **✍️ À toi de conclure** te demande de **traduire les chiffres en une phrase** pour le métier. C'est la partie la plus importante du travail d'analyste.

> **Source des données** : *DataCo Smart Supply Chain for Big Data Analysis*, F. Constante, F. Silva, A. Pereira (2019), Mendeley Data, DOI [10.17632/8gx2fvg2k6](https://data.mendeley.com/datasets/8gx2fvg2k6), licence CC BY 4.0. Le fichier utilisé ici est une version allégée : 27 colonnes sur 53, sans les données personnelles des clients, réencodée en UTF-8. Le jeu de données est aussi disponible sur Kaggle.

### 📖 Le dictionnaire des colonnes

Chaque ligne est **une ligne de commande** : un produit dans une commande. Une commande de 3 produits différents occupe donc 3 lignes.

| Colonne | Contenu |
|---|---|
| `Order Id`, `Order Item Id` | numéro de la commande, numéro de la ligne de commande |
| `order date (DateOrders)`, `shipping date (DateOrders)` | date de la commande, date d'expédition |
| `Type` | moyen de paiement |
| `Order Status` | statut de la commande (`COMPLETE`, `PENDING`, `SUSPECTED_FRAUD`…) |
| `Customer Id`, `Customer Segment` | numéro et segment du client |
| `Market`, `Order Region`, `Order Country` | zone géographique de livraison |
| `Department Name`, `Category Name`, `Product Name` | rayon, catégorie et nom du produit |
| `Order Item Product Price`, `Order Item Quantity`, `Order Item Discount Rate` | prix unitaire, quantité, taux de remise |
| `Sales` | montant avant remise |
| `Order Item Total` | montant après remise |
| `Order Profit Per Order`, `Benefit per order` | bénéfice de la ligne |
| `Order Item Profit Ratio` | bénéfice rapporté au montant |
| `Shipping Mode` | mode d'expédition (`Standard Class`, `First Class`…) |
| `Days for shipment (scheduled)`, `Days for shipping (real)` | délai de livraison promis et réel, en jours |
| `Delivery Status`, `Late_delivery_risk` | statut de livraison, indicateur de retard (1 = en retard) |

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/surybang/python_data/main/data/dataco_supply_chain.csv.gz"

---

# Partie 1 : découvrir le fichier

---

## 1.1 Charger le fichier

Le fichier est compressé (`.csv.gz`) : `pd.read_csv` sait le lire directement, sans option particulière. Charge-le dans `raw` et affiche sa taille.

In [ ]:
# TODO
raw = ...
print(raw.shape)

# attendu :
# (180519, 27)

<details>
<summary>💡 Solution</summary>

```python
brut = pd.read_csv(URL)
print(brut.shape)
```
</details>

🤔 Plus de 180 000 lignes : impossible de tout lire à l'œil. C'est exactement pour ça qu'on fait une analyse exploratoire.

> 💡 Le fichier original de Kaggle est encodé en *latin-1*. Si tu le télécharges toi-même, `pd.read_csv` lèvera une `UnicodeDecodeError` : ajoute alors `encoding="latin-1"`.

---

## 1.2 Le premier tour

Compte le nombre **total** de valeurs manquantes dans le fichier, puis le nombre de colonnes de chaque type.

In [ ]:
# TODO

# attendu :
# 0
# str        13
# int64       7
# float64     7
# Name: count, dtype: int64

<details>
<summary>💡 Solution</summary>

```python
print(brut.isna().sum().sum())
print(brut.dtypes.value_counts())
```
</details>

Aucune valeur manquante : le fichier est remarquablement complet (c'est rare, et c'est aussi parce que la version allégée a retiré les colonnes les plus trouées, comme le code postal). Mais les deux colonnes de dates sont du **texte** (`str`) : il faudra les convertir.

---

## 1.3 Des colonnes en double ?

Le dictionnaire décrit deux colonnes de bénéfice, `Order Profit Per Order` et `Benefit per order`. Vérifie si elles sont **identiques** sur toutes les lignes.

Indice : compare-les avec `==`, puis demande si **toutes** les valeurs sont vraies avec `.all()`.

In [ ]:
# TODO

# attendu :
# True

<details>
<summary>💡 Solution</summary>

```python
print((brut["Order Profit Per Order"] == brut["Benefit per order"]).all())
```
</details>

Elles sont identiques : l'une des deux est inutile. Les exports d'ERP contiennent souvent des colonnes en double, sous des noms différents. Inutile de garder les deux.

---

## 1.4 Combien de commandes ?

Combien de **commandes** différentes, de **clients** et de **produits** contient le fichier ? Combien de lignes compte une commande en moyenne ?

In [ ]:
# TODO

# attendu :
# 65752 20652 118
# 2.75

<details>
<summary>💡 Solution</summary>

```python
nb_commandes = brut["Order Id"].nunique()
print(nb_commandes, brut["Customer Id"].nunique(), brut["Product Name"].nunique())
print(round(len(brut) / nb_commandes, 2))
```
</details>

65 752 commandes pour 180 519 lignes : une commande contient en moyenne près de 3 produits. Retiens-le : **compter des lignes, ce n'est pas compter des commandes**.

---

## 1.5 Les variables qualitatives

Affiche la répartition (en proportion, arrondie à 3 décimales) des modes d'expédition.

In [ ]:
# TODO

# attendu :
# Shipping Mode
# Standard Class    0.597
# Second Class      0.195
# First Class       0.154
# Same Day          0.054
# Name: proportion, dtype: float64

<details>
<summary>💡 Solution</summary>

```python
brut["Shipping Mode"].value_counts(normalize=True).round(3)
```
</details>

---

# Partie 2 : préparer les données

Les noms de colonnes avec des espaces et des parenthèses sont pénibles à écrire (`pd.col("order date (DateOrders)")`…). On va écrire une fonction de préparation, à utiliser avec `pipe`, comme `nettoyer_ventes` dans le cours.

In [ ]:
noms = {
    "Order Id": "id_commande",
    "Order Item Id": "id_ligne",
    "order date (DateOrders)": "date_commande",
    "shipping date (DateOrders)": "date_expedition",
    "Type": "paiement",
    "Order Status": "statut",
    "Customer Id": "id_client",
    "Customer Segment": "segment",
    "Market": "marche",
    "Order Region": "region",
    "Order Country": "pays",
    "Department Name": "rayon",
    "Category Name": "categorie",
    "Product Name": "produit",
    "Order Item Product Price": "prix",
    "Order Item Quantity": "quantite",
    "Order Item Discount Rate": "taux_remise",
    "Sales": "ca_brut",
    "Order Item Total": "ca_net",
    "Order Profit Per Order": "benefice",
    "Order Item Profit Ratio": "marge",
    "Shipping Mode": "mode_expedition",
    "Days for shipment (scheduled)": "delai_prevu",
    "Days for shipping (real)": "delai_reel",
    "Delivery Status": "statut_livraison",
    "Late_delivery_risk": "en_retard",
}

---

## 2.1 La fonction de préparation

Écris une fonction `preparer(df)` qui, dans une seule chaîne :

1. supprime la colonne en double `Benefit per order` ;
2. renomme les colonnes avec le dictionnaire `noms` ;
3. convertit `date_commande` et `date_expedition` en dates (le format est `mois/jour/année heure:minute`, soit `format="%m/%d/%Y %H:%M"`) ;
4. ajoute une colonne `retard_jours` : le délai réel moins le délai prévu.

Utilise-la avec `pipe` pour créer `commandes`.

Indice : `pd.to_datetime` est une **fonction**, comme `pd.cut` dans le cours. Tu as donc besoin d'une `lambda`.

In [ ]:
# TODO
def preparer(df):
    ...


commandes = brut.pipe(preparer)
print(commandes.shape)
print(commandes[["date_commande", "date_expedition", "retard_jours"]].dtypes)

# attendu :
# (180519, 27)
# date_commande      datetime64[us]
# date_expedition    datetime64[us]
# retard_jours                int64
# dtype: object

<details>
<summary>💡 Solution</summary>

```python
def preparer(df):
    """Renomme les colonnes, convertit les dates et calcule le retard."""
    return (
        df
        .drop(columns=["Benefit per order"])
        .rename(columns=noms)
        .assign(
            date_commande=lambda d: pd.to_datetime(d["date_commande"], format="%m/%d/%Y %H:%M"),
            date_expedition=lambda d: pd.to_datetime(d["date_expedition"], format="%m/%d/%Y %H:%M"),
            retard_jours=pd.col("delai_reel") - pd.col("delai_prevu"),
        )
    )


commandes = brut.pipe(preparer)
print(commandes.shape)
print(commandes[["date_commande", "date_expedition", "retard_jours"]].dtypes)
```
</details>

---

## 2.2 Vérifier la cohérence

Deux colonnes parlent du retard : `statut_livraison` et `en_retard`. Vérifie qu'elles racontent la même histoire avec un tableau croisé.

In [ ]:
# TODO

# attendu :
# en_retard              0      1
# statut_livraison               
# Advance shipping   41592      0
# Late delivery          0  98977
# Shipping canceled   7754      0
# Shipping on time   32196      0

<details>
<summary>💡 Solution</summary>

```python
pd.crosstab(commandes["statut_livraison"], commandes["en_retard"])
```
</details>

`en_retard` vaut 1 exactement pour les lignes `Late delivery`, et 0 pour tout le reste, **y compris les expéditions annulées**. Pour mesurer la ponctualité, il faudra donc **exclure** les annulations : une commande annulée n'est ni à l'heure, ni en retard.

---

## 2.3 La période couverte

Quelle est la date de la première et de la dernière commande ?

In [ ]:
# TODO

# attendu :
# 2015-01-01 00:00:00 2018-01-31 23:38:00

<details>
<summary>💡 Solution</summary>

```python
print(commandes["date_commande"].min(), commandes["date_commande"].max())
```
</details>

---

# Partie 3 : livrons-nous à l'heure ?

In [ ]:
expediees = commandes.loc[pd.col("statut_livraison") != "Shipping canceled"]

La cellule ci-dessus crée `expediees` : toutes les lignes **sauf** les expéditions annulées. C'est la base de toute cette partie.

---

## 3.1 Le taux de retard global

Quelle proportion des lignes expédiées arrive en retard ? Arrondis à 3 décimales.

Indice : la moyenne d'une colonne de 0 et de 1 est la proportion de 1.

In [ ]:
# TODO

# attendu :
# 0.573

<details>
<summary>💡 Solution</summary>

```python
print(round(expediees["en_retard"].mean(), 3))
expediees["en_retard"].value_counts(normalize=True).round(3)
```
</details>

### ✍️ À toi de conclure

Plus d'une livraison sur deux arrive en retard. Est-ce que ça te semble acceptable pour un e-commerçant ? Que voudrais-tu savoir ensuite pour comprendre d'où vient le problème ?

*Écris ta réponse ici, en deux ou trois phrases, comme si tu l'envoyais à ton responsable.*

---

## 3.2 Par mode d'expédition

Pour chaque mode d'expédition, calcule :

- `nb_lignes` : le nombre de lignes ;
- `taux_retard` : la proportion de retards ;
- `delai_prevu` et `delai_reel` : les délais moyens, en jours.

Arrondis à 2 décimales et trie par taux de retard décroissant.

In [ ]:
# TODO
(
    expediees
    ...
)

# attendu :
#                  nb_lignes  taux_retard  delai_prevu  delai_reel
# mode_expedition                                                 
# First Class          26513         1.00          1.0        2.00
# Second Class         33806         0.80          2.0        3.99
# Same Day              9293         0.48          0.0        0.48
# Standard Class      103153         0.40          4.0        3.99

<details>
<summary>💡 Solution</summary>

```python
(
    expediees
    .groupby("mode_expedition")
    .agg(
        nb_lignes=("id_ligne", "count"),
        taux_retard=("en_retard", "mean"),
        delai_prevu=("delai_prevu", "mean"),
        delai_reel=("delai_reel", "mean"),
    )
    .round(2)
    .sort_values("taux_retard", ascending=False)
)
```
</details>

🤔 Regarde bien les colonnes de délais. Le `First Class` promet une livraison en **1 jour** et livre en **2 jours**, à chaque fois : 100 % de retards. Le `Second Class` promet 2 jours et en met 4. Le `Standard Class` promet 4 jours et livre en 4 jours **en moyenne**, mais avec de la variabilité : 40 % de ses livraisons dépassent quand même le délai.

### ✍️ À toi de conclure

Le problème des retards vient-il des transporteurs (qui livreraient lentement) ou de la **promesse** faite au client ? Quelle recommandation ferais-tu à la direction commerciale ?

*Écris ta réponse ici, en deux ou trois phrases, comme si tu l'envoyais à ton responsable.*

---

## 3.3 Le retard dépend-il de la géographie ?

Calcule le taux de retard (arrondi à 3 décimales) par marché.

In [ ]:
# TODO

# attendu :
# marche
# Africa          0.568
# Europe          0.577
# LATAM           0.569
# Pacific Asia    0.574
# USCA            0.574
# Name: en_retard, dtype: float64

<details>
<summary>💡 Solution</summary>

```python
(
    expediees
    .groupby("marche")
    ["en_retard"]
    .mean()
    .round(3)
)
```
</details>

### ✍️ À toi de conclure

Les cinq marchés ont presque exactement le même taux de retard. Qu'est-ce que ça t'apprend sur l'origine du problème ? Pourquoi est-ce un résultat aussi utile qu'une grosse différence ?

*Écris ta réponse ici, en deux ou trois phrases, comme si tu l'envoyais à ton responsable.*

---

## 3.4 Retard et mode, croisés

Avec un tableau croisé dynamique, affiche le taux de retard par marché (en lignes) et par mode d'expédition (en colonnes), arrondi à 2 décimales.

In [ ]:
# TODO

# attendu :
# mode_expedition  First Class  Same Day  Second Class  Standard Class
# marche                                                              
# Africa                   1.0      0.46          0.80             0.4
# Europe                   1.0      0.49          0.80             0.4
# LATAM                    1.0      0.50          0.79             0.4
# Pacific Asia             1.0      0.45          0.80             0.4
# USCA                     1.0      0.46          0.80             0.4

<details>
<summary>💡 Solution</summary>

```python
expediees.pivot_table(
    index="marche",
    columns="mode_expedition",
    values="en_retard",
    aggfunc="mean",
).round(2)
```
</details>

Dans **chaque** marché, on retrouve le même classement : le mode d'expédition explique les retards, la géographie non.

---

# Partie 4 : quels produits comptent vraiment ? L'analyse ABC

L'**analyse ABC** classe les produits selon leur poids dans le chiffre d'affaires :

- **A** : les produits qui font les **80 %** premiers du chiffre d'affaires, à surveiller de près (stock, prévisions) ;
- **B** : les **15 %** suivants ;
- **C** : les **5 %** restants, souvent nombreux et peu contributeurs.

C'est la loi de Pareto appliquée au stock. La méthode : trier les produits du plus gros au plus petit chiffre d'affaires, puis calculer la **part cumulée**.

---

## 4.1 Le chiffre d'affaires par produit

Crée `abc` : une ligne par produit, avec son chiffre d'affaires net (`ca`, somme de `ca_net`), trié du plus gros au plus petit, avec un index renuméroté (`reset_index()`). Affiche les 3 premières lignes et le nombre de produits.

In [ ]:
# TODO
abc = (
    commandes
    ...
)
print(abc.head(3).round(0))
print(len(abc))

# attendu :
#                                          produit         ca
# 0      Field & Stream Sportsman 16 Gun Fire Safe  6226935.0
# 1               Perfect Fitness Perfect Rip Deck  3973180.0
# 2  Diamondback Women's Serene Classic Comfort Bi  3700784.0
# 118

<details>
<summary>💡 Solution</summary>

```python
abc = (
    commandes
    .groupby("produit", as_index=False)
    .agg(ca=("ca_net", "sum"))
    .sort_values("ca", ascending=False)
    .reset_index(drop=True)
)
print(abc.head(3).round(0))
print(len(abc))
```
</details>

---

## 4.2 La part cumulée et la classe

Ajoute à `abc` :

- `part` : la part du produit dans le chiffre d'affaires total ;
- `part_cumulee` : la somme cumulée de `part` (méthode `cumsum()`) ;
- `classe` : `"A"` si la part cumulée est au plus 0,80, `"B"` si elle est au plus 0,95, `"C"` sinon (avec `case_when`).

Compte ensuite le nombre de produits de chaque classe.

In [ ]:
# TODO
abc = (
    abc
    ...
)
print(abc["classe"].value_counts().sort_index())

# attendu :
# classe
# A     7
# B    16
# C    95
# Name: count, dtype: int64

<details>
<summary>💡 Solution</summary>

```python
abc = (
    abc
    .assign(
        part=pd.col("ca") / pd.col("ca").sum(),
        part_cumulee=pd.col("part").cumsum(),
        classe=pd.col("part_cumulee").case_when([
            (pd.col("part_cumulee") <= 0.80, "A"),
            (pd.col("part_cumulee") <= 0.95, "B"),
            (pd.col("part_cumulee") > 0.95, "C"),
        ]),
    )
)
print(abc["classe"].value_counts().sort_index())
```
</details>

### ✍️ À toi de conclure

Combien de produits font 80 % du chiffre d'affaires, sur 118 ? Qu'est-ce que ça change pour la gestion des stocks et des prévisions ?

*Écris ta réponse ici, en deux ou trois phrases, comme si tu l'envoyais à ton responsable.*

---

## 4.3 Les produits de classe A

Affiche les produits de classe A, avec leur chiffre d'affaires (arrondi à l'euro) et leur part cumulée (arrondie à 3 décimales).

In [ ]:
# TODO

# attendu :
#                                          produit         ca  part_cumulee
# 0      Field & Stream Sportsman 16 Gun Fire Safe  6226935.0         0.188
# 1               Perfect Fitness Perfect Rip Deck  3973180.0         0.309
# 2  Diamondback Women's Serene Classic Comfort Bi  3700784.0         0.421
# 3              Nike Men's Free 5.0+ Running Shoe  3295693.0         0.520
# 4           Nike Men's Dri-FIT Victory Golf Polo  2828708.0         0.606
# 5                    Pelican Sunstream 100 Kayak  2785518.0         0.690
# 6        Nike Men's CJ Elite 2 TD Football Cleat  2598494.0         0.769

<details>
<summary>💡 Solution</summary>

```python
(
    abc
    .loc[pd.col("classe") == "A", ["produit", "ca", "part_cumulee"]]
    .round({"ca": 0, "part_cumulee": 3})
)
```
</details>

### 📊 Bonus : la courbe de Pareto

Pandas sait tracer des graphiques simples avec `.plot()` (il s'appuie sur la bibliothèque matplotlib). Ce n'est pas l'objet du module, mais la courbe de Pareto vaut le détour. Exécute simplement la cellule :

In [ ]:
ax = abc["part_cumulee"].plot(
    title="Part cumulée du chiffre d'affaires",
    xlabel="Produits, du plus vendu au moins vendu",
    ylabel="Part cumulée",
)
ax.axhline(0.8, linestyle="--", color="grey");

---

# Partie 5 : où gagne-t-on, où perd-on de l'argent ?

---

## 5.1 Des lignes à perte

Quelle proportion des lignes de commande a un bénéfice **négatif** ? Arrondis à 3 décimales.

In [ ]:
# TODO

# attendu :
# 0.187

<details>
<summary>💡 Solution</summary>

```python
print(round((commandes["benefice"] < 0).mean(), 3))
```
</details>

---

## 5.2 Le bénéfice par rayon

Pour chaque rayon, calcule le chiffre d'affaires net (`ca`), le bénéfice total (`benefice`) et le **taux de marge** global (`taux_marge` = bénéfice total divisé par chiffre d'affaires total). Arrondis et trie par taux de marge croissant.

Attention : on veut le rapport des **totaux**, pas la moyenne de la colonne `marge` (souviens-toi de la moyenne des moyennes).

In [ ]:
# TODO
(
    commandes
    ...
)

# attendu :
#                             ca   benefice  taux_marge
# rayon                                                
# Book Shop              11303.0      883.0       0.078
# Pet Shop               37318.0     3589.0       0.096
# Health and Beauty      95358.0     9494.0       0.100
# Footwear             3599469.0   410222.0       0.114
# Discs Shop            205676.0    24193.0       0.118
# Fan Shop            15378582.0  1834155.0       0.119
# Golf                 4142101.0   497524.0       0.120
# Technology            933829.0   113170.0       0.121
# Apparel              7167654.0   881883.0       0.123
# Outdoors             1125953.0   145251.0       0.129
# Fitness               357158.0    46538.0       0.130

<details>
<summary>💡 Solution</summary>

```python
(
    commandes
    .groupby("rayon")
    .agg(ca=("ca_net", "sum"), benefice=("benefice", "sum"))
    .assign(taux_marge=pd.col("benefice") / pd.col("ca"))
    .round({"ca": 0, "benefice": 0, "taux_marge": 3})
    .sort_values("taux_marge")
)
```
</details>

---

## 5.3 Les remises détruisent-elles la marge ?

Découpe le taux de remise en tranches avec `pd.cut` : `[0, 0.05, 0.10, 0.15, 0.25]` (avec `include_lowest=True` pour garder les remises nulles). Pour chaque tranche, calcule le nombre de lignes et le taux de marge global (bénéfice total / CA net total), arrondi à 3 décimales.

In [ ]:
# TODO
(
    commandes
    ...
)

# attendu :
# tranche_remise
# (-0.001, 0.05]    0.120
# (0.05, 0.1]       0.124
# (0.1, 0.15]       0.119
# (0.15, 0.25]      0.118
# Name: taux_marge, dtype: float64

<details>
<summary>💡 Solution</summary>

```python
(
    commandes
    .assign(tranche_remise=lambda d: pd.cut(d["taux_remise"], bins=[0, 0.05, 0.10, 0.15, 0.25], include_lowest=True))
    .groupby("tranche_remise", observed=True)
    .agg(nb_lignes=("id_ligne", "count"), ca=("ca_net", "sum"), benefice=("benefice", "sum"))
    .assign(taux_marge=pd.col("benefice") / pd.col("ca"))
    ["taux_marge"]
    .round(3)
)
```
</details>

### ✍️ À toi de conclure

La remise fait-elle beaucoup varier le taux de marge ? Si les lignes à perte ne s'expliquent pas par les remises, quelle autre piste regarderais-tu (indice : le bénéfice est-il le même pour toutes les lignes d'un même produit) ?

*Écris ta réponse ici, en deux ou trois phrases, comme si tu l'envoyais à ton responsable.*

---

# Partie 6 : y a-t-il de la fraude ?

---

## 6.1 Le poids de la fraude

Combien de **commandes** (pas de lignes !) ont le statut `SUSPECTED_FRAUD`, et quel chiffre d'affaires net représentent-elles (arrondi à l'euro) ?

In [ ]:
# TODO
fraude = commandes.loc[...]

# attendu :
# 1488
# 741492

<details>
<summary>💡 Solution</summary>

```python
fraude = commandes.loc[pd.col("statut") == "SUSPECTED_FRAUD"]
print(fraude["id_commande"].nunique())
print(round(fraude["ca_net"].sum()))
```
</details>

---

## 6.2 Fraude et moyen de paiement

Avec `pd.crosstab`, calcule pour chaque moyen de paiement la part des lignes suspectes de fraude. Arrondis à 3 décimales.

Indice : crée d'abord une colonne booléenne `suspecte`.

In [ ]:
# TODO

# attendu :
# suspecte  False  True 
# paiement              
# CASH      1.000  0.000
# DEBIT     1.000  0.000
# PAYMENT   1.000  0.000
# TRANSFER  0.919  0.081

<details>
<summary>💡 Solution</summary>

```python
(
    commandes
    .assign(suspecte=pd.col("statut") == "SUSPECTED_FRAUD")
    .pipe(lambda d: pd.crosstab(d["paiement"], d["suspecte"], normalize="index"))
    .round(3)
)
```
</details>

### ✍️ À toi de conclure

Que remarques-tu ? Quelle règle simple proposerais-tu au service fraude ? Et quelle question poserais-tu sur la façon dont ce statut a été attribué (est-ce une fraude prouvée ou un soupçon) ?

*Écris ta réponse ici, en deux ou trois phrases, comme si tu l'envoyais à ton responsable.*

---

# Partie 7 : comment l'activité évolue-t-elle ?

---

## 7.1 Les commandes mois par mois

Crée `mensuel` : pour chaque mois (`to_period("M")`), le nombre de **commandes** distinctes et le chiffre d'affaires net. Affiche les 8 derniers mois, avec le CA arrondi à l'euro.

In [ ]:
# TODO
mensuel = (
    commandes
    ...
)
print(mensuel.tail(8))

# attendu :
#          nb_commandes         ca
# mois                            
# 2017-06          1676   927446.0
# 2017-07          1776   992423.0
# 2017-08          1768   996508.0
# 2017-09          1723  1027112.0
# 2017-10          2101   966081.0
# 2017-11          2055   563393.0
# 2017-12          2124   452742.0
# 2018-01          2123   297952.0

<details>
<summary>💡 Solution</summary>

```python
mensuel = (
    commandes
    .assign(mois=pd.col("date_commande").dt.to_period("M"))
    .groupby("mois")
    .agg(nb_commandes=("id_commande", "nunique"), ca=("ca_net", "sum"))
    .round(0)
)
print(mensuel.tail(8))
```
</details>

In [ ]:
mensuel["ca"].plot(title="Chiffre d'affaires net par mois");

### ✍️ À toi de conclure

À partir d'octobre 2017, le nombre de commandes reste stable (il augmente même), mais le chiffre d'affaires mensuel s'effondre, de 1 million à 300 000. Avant d'annoncer une chute des ventes à la direction, quelles vérifications ferais-tu (changement de périmètre, de système, données incomplètes) ? Pourquoi est-il risqué de calculer une tendance « sur toute la période » ?

*Écris ta réponse ici, en deux ou trois phrases, comme si tu l'envoyais à ton responsable.*

---

## 7.2 Pour aller plus loin : le panier moyen a-t-il changé ?

Compare, avant et après le 1er octobre 2017, le nombre moyen de **lignes** par commande et le chiffre d'affaires net moyen **par commande**.

Indice : regroupe d'abord par commande, puis par période.

In [ ]:
# TODO

# attendu :
#          nb_lignes_moyen  panier_moyen
# periode                               
# après               1.02        271.35
# avant               3.00        536.61

<details>
<summary>💡 Solution</summary>

```python
(
    commandes
    .groupby("id_commande", as_index=False)
    .agg(date=("date_commande", "first"), nb_lignes=("id_ligne", "count"), ca=("ca_net", "sum"))
    .assign(periode=pd.col("date").case_when([
        (pd.col("date") < "2017-10-01", "avant"),
        (pd.col("date") >= "2017-10-01", "après"),
    ]))
    .groupby("periode")
    .agg(nb_lignes_moyen=("nb_lignes", "mean"), panier_moyen=("ca", "mean"))
    .round(2)
)
```
</details>

Après octobre 2017, les commandes ne contiennent plus qu'**une seule** ligne. Ce n'est pas un comportement client : c'est presque certainement un changement dans la façon dont les données ont été enregistrées. Un bon réflexe d'analyste : quand une courbe casse brutalement, suspecter **les données** avant le métier.

---

# Partie 8 : la synthèse

Une analyse exploratoire se termine toujours par une **restitution**. Rédige ci-dessous une note de 5 à 10 lignes pour ton responsable :

- les **3 constats** les plus importants, chacun appuyé par un chiffre ;
- pour chacun, une **recommandation** ou une **question** à creuser avec le métier ;
- les **limites** des données que tu as rencontrées.

Pas de jargon technique : ton responsable ne lit pas de code.

### ✍️ Ma note de synthèse

*1. Constat :*

*2. Constat :*

*3. Constat :*

*Limites des données :*

<details>
<summary>💡 Un exemple de synthèse</summary>

1. **Les retards viennent surtout de nos promesses.** 57 % des livraisons arrivent en retard. Le First Class promet 1 jour et livre toujours en 2 (100 % de retards), le Second Class promet 2 jours et en met 4 (80 %). Recommandation : aligner les délais affichés sur les délais réels, ce qui supprimerait une grande partie des retards sans coût logistique. Le Standard Class respecte son délai en moyenne mais reste en retard 4 fois sur 10 : là, c'est la régularité du transport qu'il faut travailler. La géographie n'y est pour rien : tous les marchés sont autour de 57 %.
2. **Une poignée de produits fait l'essentiel du chiffre d'affaires.** 7 produits sur 118 font 80 % du CA. Ce sont eux qu'il faut prévoir finement et ne jamais laisser en rupture.
3. **La fraude suspectée ne touche que les virements.** Environ 8 % des lignes payées par virement sont suspectes, contre 0 % pour les autres moyens de paiement. À creuser avec le service fraude : est-ce une vraie fraude ou une règle de détection appliquée aux seuls virements ?

**Limites** : à partir d'octobre 2017, les commandes ne comptent plus qu'une ligne et le chiffre d'affaires mensuel est divisé par trois alors que le nombre de commandes est stable, ce qui ressemble à un changement de système. Les comparaisons dans le temps doivent s'arrêter à septembre 2017 tant que ce point n'est pas éclairci. Près d'une ligne sur cinq est vendue à perte, sans lien clair avec les remises : à étudier avec le contrôle de gestion.
</details>